# Half-SLM Parameter Sweep

Sweep target size, target smoothing, and initial curvature using one slmsuite calibration file and one half of the SLM.

## 1. Imports and Sweep Parameters

In [ ]:
from itertools import product
from time import perf_counter

import numpy as np
import matplotlib.pyplot as plt

from initial_holograms import curvature_hologram
from loss_functions import build_circular_mask
from optimizer import CGOptimizer
from optical_planes import CameraPlane, SLMPlane
from propagator import Propagator
from target_profiles import apply_psf_smoothing, build_rectangle_target

# Native SLM and half-SLM compute grid.
full_slm_shape = (1080, 1920)
superpixel_size = 4
slm_shape = (full_slm_shape[0] // superpixel_size, full_slm_shape[1] // superpixel_size)
half_slm_shape = (slm_shape[0], slm_shape[1] // 2)
padding_factor = 2
half_camera_shape = (half_slm_shape[0] * padding_factor, half_slm_shape[1] * padding_factor)

# Optical parameters.
wavelength_nm = 420.0
slm_pixel_pitch_um = 8.0
camera_pixel_pitch_um = 3.45
focal_length_mm = 200.0

# One slmsuite wavefront-calibration file.
calibration_h5_path = "10806-SLM-wavefront_superpixel-calibration_00036.h5"
calibration_side = "left"
slmsuite_camera_shape = full_slm_shape
slmsuite_wavefront_r2_threshold = 0.5
slmsuite_remove_background = True
slmsuite_apply_calibration = True
slmsuite_amplitude_key = "amplitude"
slmsuite_phase_key = "phase"

# Parameter sweep definitions.
# Rectangle entries are (width_x_um, height_y_um), using a vertical rectangle by default.
scan_rectangle_sizes_um = [
    (20.0, 250.0),
    (30.0, 300.0),
    (40.0, 350.0),
]
scan_psf_sigma_um = [5.0, 10.0, 15.0]
scan_quadratic_curvature = [1.5e-3, 2.5e-3, 3.6e-3, 4.5e-3]

# Fixed mask and hologram parameters.
mask_radius_margin_um = 100.0
initial_phase_linear_tilt = 0.0
initial_phase_astigmatism_weight = 0.0
initial_phase_linear_angle_rad = np.pi / 4.0
initial_phase_conical_weight = 0.0

# CG parameters for each scan point.
optimizer_maxiter = 80
loss_scale = 1e12
optimize_phase = True

# Ranking criterion. Increase fidelity_weight if phase/amplitude fidelity matters more than throughput.
fidelity_weight = 0.5
efficiency_weight = 0.5


## 2. Load Calibration File

In [ ]:
def downsample_by_superpixel(data: np.ndarray, superpixel_size: int) -> np.ndarray:
    data_array = np.asarray(data, dtype=float)
    if data_array.ndim != 2:
        raise ValueError(f"data must be 2D, got shape {data_array.shape}.")
    if data_array.shape[0] % superpixel_size != 0 or data_array.shape[1] % superpixel_size != 0:
        raise ValueError(f"data shape must be divisible by superpixel_size, got data_shape={data_array.shape}, superpixel_size={superpixel_size}.")
    ny = data_array.shape[0] // superpixel_size
    nx = data_array.shape[1] // superpixel_size
    return data_array.reshape(ny, superpixel_size, nx, superpixel_size).mean(axis=(1, 3))


def downsample_phase_by_superpixel(phase: np.ndarray, superpixel_size: int) -> np.ndarray:
    phase_array = np.asarray(phase, dtype=float)
    if phase_array.ndim != 2:
        raise ValueError(f"phase must be 2D, got shape {phase_array.shape}.")
    real_part = downsample_by_superpixel(np.cos(phase_array), superpixel_size)
    imag_part = downsample_by_superpixel(np.sin(phase_array), superpixel_size)
    return np.angle(real_part + 1j * imag_part)


def require_calibration_array(calibration_results: dict, key: str, expected_shape: tuple[int, int]) -> np.ndarray:
    if key not in calibration_results:
        available_keys = sorted(str(item) for item in calibration_results.keys())
        raise KeyError(f"Calibration result key {key!r} was not found. Available keys: {available_keys}.")
    array = np.asarray(calibration_results[key], dtype=float)
    if array.shape != expected_shape:
        raise ValueError(f"Calibration result {key!r} shape must be {expected_shape}, got {array.shape}.")
    if not np.all(np.isfinite(array)):
        raise ValueError(f"Calibration result {key!r} contains NaN or infinite values.")
    return array


def load_slmsuite_input_beam_and_phase(calibration_h5_path: str, full_slm_shape: tuple[int, int], superpixel_size: int, slm_pixel_pitch_um: float, wavelength_nm: float, camera_shape: tuple[int, int], r2_threshold: float, remove_background: bool, apply_calibration: bool, amplitude_key: str, phase_key: str) -> tuple[np.ndarray, np.ndarray, dict]:
    from slmsuite.hardware.cameras.simulated import SimulatedCamera
    from slmsuite.hardware.cameraslms import FourierSLM
    from slmsuite.hardware.slms.simulated import SimulatedSLM

    slm_size_xy = (int(full_slm_shape[1]), int(full_slm_shape[0]))
    camera_size_xy = (int(camera_shape[1]), int(camera_shape[0]))
    slm = SimulatedSLM(slm_size_xy, pitch_um=(slm_pixel_pitch_um, slm_pixel_pitch_um), wav_um=wavelength_nm / 1000.0)
    camera = SimulatedCamera(slm, resolution=camera_size_xy)
    fourier_slm = FourierSLM(camera, slm)
    fourier_slm.load_calibration("wavefront_superpixel", file_path=calibration_h5_path)
    calibration_results = fourier_slm.wavefront_calibration_superpixel_process(
        plot=False,
        r2_threshold=r2_threshold,
        remove_background=remove_background,
        apply=apply_calibration,
    )
    amplitude_full = require_calibration_array(calibration_results, amplitude_key, full_slm_shape)
    phase_full = require_calibration_array(calibration_results, phase_key, full_slm_shape)
    amplitude = downsample_by_superpixel(np.clip(amplitude_full, 0.0, None), superpixel_size)
    phase = downsample_phase_by_superpixel(phase_full, superpixel_size)
    if np.max(amplitude) <= 0:
        raise ValueError("Loaded slmsuite amplitude contains no positive values after downsampling.")
    return amplitude / np.max(amplitude), np.mod(phase, 2.0 * np.pi), calibration_results


def crop_half_slm(amplitude: np.ndarray, phase: np.ndarray, side: str) -> tuple[np.ndarray, np.ndarray]:
    if side not in ("left", "right"):
        raise ValueError(f"side must be 'left' or 'right', got {side}.")
    amplitude_array = np.asarray(amplitude, dtype=float)
    phase_array = np.asarray(phase, dtype=float)
    if amplitude_array.shape != slm_shape or phase_array.shape != slm_shape:
        raise ValueError(f"amplitude and phase must both have shape {slm_shape}, got {amplitude_array.shape} and {phase_array.shape}.")
    if slm_shape[1] % 2 != 0:
        raise ValueError(f"SLM x size must be even for a left/right split, got {slm_shape[1]}.")
    split_index = slm_shape[1] // 2
    if side == "left":
        amplitude_half = amplitude_array[:, :split_index]
        phase_half = phase_array[:, :split_index]
    else:
        amplitude_half = amplitude_array[:, split_index:]
        phase_half = phase_array[:, split_index:]
    if np.max(amplitude_half) <= 0:
        raise ValueError(f"{side} half input amplitude contains no positive values.")
    return amplitude_half / np.max(amplitude_half), phase_half.copy()


full_input_amplitude, full_input_phase, calibration_results = load_slmsuite_input_beam_and_phase(
    calibration_h5_path,
    full_slm_shape,
    superpixel_size,
    slm_pixel_pitch_um,
    wavelength_nm,
    slmsuite_camera_shape,
    slmsuite_wavefront_r2_threshold,
    slmsuite_remove_background,
    slmsuite_apply_calibration,
    slmsuite_amplitude_key,
    slmsuite_phase_key,
)
input_amplitude, input_phase = crop_half_slm(full_input_amplitude, full_input_phase, calibration_side)

print("Full superpixel SLM shape:", slm_shape)
print("Half superpixel SLM shape:", half_slm_shape)
print("Selected half:", calibration_side)
print("Half input shape:", input_amplitude.shape)

fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.0), constrained_layout=True)
axes[0].imshow(input_amplitude**2, origin="lower", cmap="magma", aspect="equal")
axes[0].set_title("Half-SLM input intensity")
axes[1].imshow(input_phase, origin="lower", cmap="twilight", aspect="equal")
axes[1].set_title("Half-SLM input phase")
plt.show()


## 3. Run Optimization Sweep

In [ ]:
def build_optimizer_for_scan_point(width_x_um: float, height_y_um: float, psf_sigma_um: float, quadratic_curvature: float) -> CGOptimizer:
    initial_hologram = curvature_hologram(
        shape=half_slm_shape,
        linear_tilt=initial_phase_linear_tilt,
        astigmatism_weight=initial_phase_astigmatism_weight,
        quadratic_curvature=quadratic_curvature,
        linear_angle_rad=initial_phase_linear_angle_rad,
        conical_weight=initial_phase_conical_weight,
    )
    slm = SLMPlane(half_slm_shape, wavelength_nm, slm_pixel_pitch_um, superpixel_size, input_amplitude, input_phase, initial_hologram)
    camera = CameraPlane(half_camera_shape, wavelength_nm, (1.0, 1.0), camera_pixel_pitch_um, np.zeros(half_camera_shape), np.zeros(half_camera_shape))
    propagator = Propagator(slm, camera, focal_length_mm, padding_factor)
    ideal_target = build_rectangle_target(
        half_camera_shape,
        propagator.camera_plane.x_axis_um,
        propagator.camera_plane.y_axis_um,
        width_x_um,
        height_y_um,
    )
    target_amplitude = apply_psf_smoothing(ideal_target, psf_sigma_um, psf_sigma_um, propagator.camera_plane.scale_um)
    target_phase = np.zeros_like(target_amplitude)
    mask_center_x_um = 0.5 * (propagator.camera_plane.x_axis_um[0] + propagator.camera_plane.x_axis_um[-1])
    mask_center_y_um = 0.5 * (propagator.camera_plane.y_axis_um[0] + propagator.camera_plane.y_axis_um[-1])
    mask_radius_um = max(width_x_um, height_y_um) / 2.0 + mask_radius_margin_um
    target_mask = build_circular_mask(
        half_camera_shape,
        propagator.camera_plane.x_axis_um,
        propagator.camera_plane.y_axis_um,
        mask_center_x_um,
        mask_center_y_um,
        mask_radius_um,
    )
    target_plane = CameraPlane(half_camera_shape, wavelength_nm, propagator.camera_plane.scale_um, camera_pixel_pitch_um, target_amplitude, target_phase)
    optimizer = CGOptimizer(propagator, target_plane, target_mask)
    optimizer.set_initial_hologram_array(initial_hologram)
    return optimizer


scan_results = []
best_optimizer = None
best_score = -np.inf
scan_start = perf_counter()
scan_points = list(product(scan_rectangle_sizes_um, scan_psf_sigma_um, scan_quadratic_curvature))
print("Total scan points:", len(scan_points))

for scan_index, (rectangle_size_um, psf_sigma_um, quadratic_curvature) in enumerate(scan_points, start=1):
    width_x_um, height_y_um = rectangle_size_um
    print(f"[{scan_index}/{len(scan_points)}] width_x={width_x_um:.3g} um, height_y={height_y_um:.3g} um, psf={psf_sigma_um:.3g} um, curvature={quadratic_curvature:.3g}")
    optimizer = build_optimizer_for_scan_point(width_x_um, height_y_um, psf_sigma_um, quadratic_curvature)
    optimizer.optimize(
        maxiter=optimizer_maxiter,
        loss_scale=loss_scale,
        optimize_phase=optimize_phase,
    )
    result = optimizer.get_result_summary()
    score = fidelity_weight * result.fidelity + efficiency_weight * result.efficiency
    entry = {
        "scan_index": scan_index,
        "width_x_um": width_x_um,
        "height_y_um": height_y_um,
        "psf_sigma_um": psf_sigma_um,
        "quadratic_curvature": quadratic_curvature,
        "efficiency": result.efficiency,
        "fidelity": result.fidelity,
        "rms_error": result.rms_error,
        "phase_error": result.phase_error,
        "score": score,
        "loss_final": optimizer.loss_history[-1] if optimizer.loss_history else np.nan,
        "optimizer": optimizer,
    }
    scan_results.append(entry)
    if score > best_score:
        best_score = score
        best_optimizer = optimizer

scan_time_sec = perf_counter() - scan_start
print("Scan finished in seconds:", scan_time_sec)
print("Best score:", best_score)


## 4. Show Sweep Results

In [ ]:
def scan_array(key: str) -> np.ndarray:
    return np.asarray([entry[key] for entry in scan_results], dtype=float)


if not scan_results:
    raise RuntimeError("scan_results is empty. Run the scan block first.")

sorted_results = sorted(scan_results, key=lambda entry: entry["score"], reverse=True)
print("Top scan results:")
for entry in sorted_results[:10]:
    print(
        f"#{entry['scan_index']:03d} "
        f"score={entry['score']:.6g}, fidelity={entry['fidelity']:.6g}, efficiency={entry['efficiency']:.6g}, "
        f"rms={entry['rms_error']:.6g}, phase={entry['phase_error']:.6g}, "
        f"width_x={entry['width_x_um']:.3g}, height_y={entry['height_y_um']:.3g}, "
        f"psf={entry['psf_sigma_um']:.3g}, curvature={entry['quadratic_curvature']:.3g}"
    )

best_entry = sorted_results[0]
best_optimizer = best_entry["optimizer"]
best_result = best_optimizer.get_result_summary()
best_hologram = best_optimizer.get_final_hologram()
best_full_resolution_hologram = best_optimizer.get_full_resolution_hologram()

fig, axes = plt.subplots(1, 3, figsize=(15.0, 4.2), constrained_layout=True)
scatter0 = axes[0].scatter(scan_array("efficiency"), scan_array("fidelity"), c=scan_array("score"), cmap="viridis", s=60)
axes[0].set_xlabel("Efficiency")
axes[0].set_ylabel("Fidelity")
axes[0].set_title("Fidelity vs efficiency")
axes[0].grid(True, alpha=0.25, linestyle="--")
fig.colorbar(scatter0, ax=axes[0], label="score")

scatter1 = axes[1].scatter(scan_array("quadratic_curvature"), scan_array("fidelity"), c=scan_array("efficiency"), cmap="plasma", s=60)
axes[1].set_xlabel("Quadratic curvature")
axes[1].set_ylabel("Fidelity")
axes[1].set_title("Curvature sweep")
axes[1].grid(True, alpha=0.25, linestyle="--")
fig.colorbar(scatter1, ax=axes[1], label="efficiency")

scatter2 = axes[2].scatter(scan_array("psf_sigma_um"), scan_array("efficiency"), c=scan_array("fidelity"), cmap="magma", s=60)
axes[2].set_xlabel("PSF sigma (um)")
axes[2].set_ylabel("Efficiency")
axes[2].set_title("PSF smoothing sweep")
axes[2].grid(True, alpha=0.25, linestyle="--")
fig.colorbar(scatter2, ax=axes[2], label="fidelity")
plt.show()

print("Best full-resolution hologram shape:", best_full_resolution_hologram.shape)
print("Best superpixel hologram shape:", best_hologram.shape)
best_optimizer.plot_result_summary()
plt.show()
